# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | حسين الحمادة — add all group members |
| **Student ID** | [Group student IDs required] |
| **Section** | [Section required] |
| **Date submitted** | [Actual submission date required] |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

**Run provenance:** The saved measurements were produced in an assistant execution environment, not on the group's personal computer. Re-run on your course `.venv` before submission. Analysis below is an evidence-based draft; review and rewrite personal reflections in your own words.


---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import os
# Explicit, reproducible worker limit; record it with the results.
os.environ.setdefault("POLARS_MAX_THREADS", "4")
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)
from IPython.display import display
print("Polars worker threads:", pl.thread_pool_size())


pandas: 2.2.3
polars: 2.0.0
Polars worker threads: 4


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

**Chosen dataset:** nycflights13, all 336,776 departures from NYC in 2013. This CSV mirror includes a `rownames` identifier and 19 original columns. `carrier` is the category; `arr_delay` and `dep_delay` are both measured in minutes. `new_col = arr_delay - dep_delay` represents the change in delay during the flight. Missing measurements stay missing; filters exclude missing arrival delays, and sums/means ignore missing numeric values. No synthetic rows or duplication are used.

[Dataset documentation](https://nycflights13.tidyverse.org/reference/flights.html) · [CSV download](https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/nycflights13/flights.csv)


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

DATA_PATH = "data/flights.csv"
DATASET_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/nycflights13/flights.csv"
DATASET_DOCS = "https://nycflights13.tidyverse.org/reference/flights.html"
# Download only when missing, outside the timed operations.
Path("data").mkdir(exist_ok=True)
if not Path(DATA_PATH).is_file():
    urlretrieve(DATASET_URL, DATA_PATH)

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")
assert rows >= 100_000
assert df_pandas.shape == df_polars.shape
print("CSV size (MiB):", round(Path(DATA_PATH).stat().st_size / 1024**2, 2))
print("SHA256:", hashlib.sha256(Path(DATA_PATH).read_bytes()).hexdigest())
print("Dataset documentation:", DATASET_DOCS)
print("CSV source:", DATASET_URL)


pandas shape: (336776, 20)
polars shape: (336776, 20)

columns: ['rownames', 'year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'time_hour']

   rownames  year  month  day  ...  distance  hour  minute             time_hour
0         1  2013      1    1  ...      1400     5      15  2013-01-01T10:00:00Z
1         2  2013      1    1  ...      1416     5      29  2013-01-01T10:00:00Z
2         3  2013      1    1  ...      1089     5      40  2013-01-01T10:00:00Z

[3 rows x 20 columns]

rows: 336,776
Size is fine.
CSV size (MiB): 31.67
SHA256: b791a2590c9167aa0bd3cb45af3f64101180878cf088c183eae4011e3a654f42
Dataset documentation: https://nycflights13.tidyverse.org/reference/flights.html
CSV source: https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/nycflights13/flights.csv


---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.022 ms   <- nothing was executed
lazy WITH .collect():      51.917 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---:|
| Lazy query without `.collect()` | 0.022 |
| Lazy query with `.collect()` | 51.917 |

The first number measures query-plan construction, not reading, filtering, or producing the result; `.collect()` is needed to execute that work.


---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "carrier"    # a text/category column, for grouping
NUMERIC_COL  = "arr_delay"     # a numeric column
NUMERIC_COL2 = "dep_delay"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")
assert pd.api.types.is_numeric_dtype(df_pandas[NUMERIC_COL])
assert pd.api.types.is_numeric_dtype(df_pandas[NUMERIC_COL2])
assert not df_pandas[CATEGORY_COL].isna().any()
print("Missing values:")
print(df_pandas[[CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]].isna().sum())


[ok] carrier
[ok] arr_delay
[ok] dep_delay
Missing values:
carrier         0
arr_delay    9430
dep_delay    8255
dtype: int64


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [6]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:   251.26 ms
polars:    51.96 ms
ratio:      4.84x


**Observation:** Pandas took 251.264 ms and Polars took 51.965 ms (4.84x Pandas/Polars). The file was already loaded before timing, and the helper adds another warm-up, so this compares warm-cache CSV parsing and allocation rather than a cold disk read or network download.

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [7]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: -5.0
pandas:    11.27 ms
polars:     4.76 ms
ratio:      2.37x


**Observation:** Pandas took 11.268 ms and Polars took 4.759 ms (2.37x Pandas/Polars). Both versions use the same -5 minute arrival-delay threshold and exclude missing delays. The independent row-ID check confirmed that they retained exactly the same rows.

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [8]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     8.26 ms
polars:     0.76 ms
ratio:     10.82x


**Observation:** Pandas took 8.262 ms and Polars took 0.763 ms (10.82x Pandas/Polars). The result is arrival delay minus departure delay, both in minutes. Pandas assign returns a new DataFrame, whereas Polars can reuse unchanged columns; that may contribute to the gap, but this experiment does not isolate its cause.

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [9]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    12.87 ms
polars:     3.72 ms
ratio:      3.46x

pandas result (first rows):
carrier
9E    127624.0
AA     11638.0
AS     -7041.0
B6    511194.0
DL     78366.0
Name: arr_delay, dtype: float64

polars result (first rows):
shape: (5, 2)
┌─────────┬───────────┐
│ carrier ┆ arr_delay │
│ ---     ┆ ---       │
│ str     ┆ i64       │
╞═════════╪═══════════╡
│ AA      ┆ 11638     │
│ EV      ┆ 807324    │
│ MQ      ┆ 269767    │
│ US      ┆ 42232     │
│ UA      ┆ 205589    │
└─────────┴───────────┘


**Observation:** Pandas took 12.871 ms and Polars took 3.723 ms (3.46x Pandas/Polars). Both sums and means agree after aligning by carrier; means agree within floating-point tolerance. Pandas sorts group keys by default, while the Polars output order can differ, so their first five displayed rows need not match.

## 4.5 — Sorting and taking the top N

In [10]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False, kind="stable").head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True, nulls_last=True, maintain_order=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    65.58 ms
polars:    20.57 ms
ratio:      3.19x


**Observation:** Pandas took 65.583 ms and Polars took 20.572 ms (3.19x Pandas/Polars). Both sorts place missing delays last and preserve input order for ties. Their top ten row identifiers agree; the measurement includes a full sort followed by head(10), rather than only a top-k selection.

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [11]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):   275.38 ms
polars (lazy + collect):    27.12 ms
ratio:    10.16x


**Look at the query plan.** Polars can show you what it decided to do.

In [12]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("arr_delay").sum()] BY [col("carrier")]
  FROM
  simple π 2/2 ["carrier", "arr_delay"]
    Csv SCAN [data/flights.csv]
    PROJECT 2/20 COLUMNS
    SELECTION: col("arr_delay").cast(Float64) > -5.0
    ESTIMATED ROWS: 368967


**Observation:** Pandas took 275.376 ms and Polars took 27.116 ms (10.16x Pandas/Polars). The printed optimized plan for the separate inspection query says PROJECT 2/20 COLUMNS and shows the arrival-delay selection inside the CSV scan. The timed pipeline needs three columns (carrier, arr_delay, dep_delay) and includes .collect(); the Pandas pipeline reads all columns. Projection and filtering in the scan are plausible contributors to the gap. The estimated row count in the plan is not the actual dataset row count.

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [13]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    27.66 ms
polars:     4.40 ms
ratio:      6.29x

rows before: 336776 -> after: 336776
row count unchanged: True


**Observation:** Pandas took 27.663 ms and Polars took 4.401 ms (6.29x Pandas/Polars). The lookup has one row per carrier. Both left joins preserve all 336,776 flight rows and attach the same category average after row alignment; lookup construction is outside the timed join.

---
# Part 5 · Results summary

Collect every measurement into one table.

In [14]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["pandas_ms"] > results["polars_ms"], "polars", "pandas")

results.round(2)


          operation  pandas_ms  polars_ms  speedup  faster
0         read file     251.26      51.96     4.84  polars
1            filter      11.27       4.76     2.37  polars
2        new column       8.26       0.76    10.82  polars
3          group by      12.87       3.72     3.46  polars
4      sort + top N      65.58      20.57     3.19  polars
5  chained pipeline     275.38      27.12    10.16  polars
6              join      27.66       4.40     6.29  polars


In [15]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Linux-6.18.44-x86_64-with-glibc2.39
   x86_64


### Verify correctness independently of timings
Results are aligned by keys before comparing. Floating-point means use a numerical tolerance.

In [16]:
# Numerical agreement is checked independently from timing.
from pandas.testing import assert_frame_equal

def to_pd(frame):
    return pd.DataFrame(frame.to_dict(as_series=False))

def assert_numeric_equal(left, right):
    np.testing.assert_allclose(np.asarray(left, dtype=float),
                               np.asarray(right, dtype=float),
                               rtol=1e-10, atol=1e-8, equal_nan=True)

assert_numeric_equal(df_pandas[NUMERIC_COL], df_polars[NUMERIC_COL].to_numpy())
assert_numeric_equal(df_pandas[NUMERIC_COL2], df_polars[NUMERIC_COL2].to_numpy())
pf = df_pandas[df_pandas[NUMERIC_COL] > threshold]
qf = df_polars.filter(pl.col(NUMERIC_COL) > threshold)
assert pf["rownames"].tolist() == qf["rownames"].to_list()
assert_numeric_equal(df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2],
                     df_polars.select((pl.col(NUMERIC_COL)-pl.col(NUMERIC_COL2)).alias("new_col"))["new_col"].to_numpy())
pg = df_pandas.groupby(CATEGORY_COL).agg(total=(NUMERIC_COL,"sum"),average=(NUMERIC_COL,"mean")).reset_index().sort_values(CATEGORY_COL)
qg = to_pd(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum().alias("total"), pl.col(NUMERIC_COL).mean().alias("average")).sort(CATEGORY_COL))
assert pg[CATEGORY_COL].tolist() == qg[CATEGORY_COL].tolist()
assert_numeric_equal(pg[["total","average"]],qg[["total","average"]])
pt = df_pandas.sort_values(NUMERIC_COL,ascending=False,kind="stable").head(10)
qt = df_polars.sort(NUMERIC_COL,descending=True,nulls_last=True,maintain_order=True).head(10)
assert pt["rownames"].tolist() == qt["rownames"].to_list()
pp = pandas_equivalent().reset_index().sort_values(CATEGORY_COL)
qp = to_pd(polars_lazy_pipeline().sort(CATEGORY_COL))
assert pp[CATEGORY_COL].tolist() == qp[CATEGORY_COL].tolist()
assert_numeric_equal(pp["total"],qp["total"])
pj = df_pandas.merge(lookup_pandas,on=CATEGORY_COL,how="left").sort_values("rownames")
qj = to_pd(df_polars.join(lookup_polars,on=CATEGORY_COL,how="left").sort("rownames"))
assert len(pj) == len(qj) == len(df_pandas)
assert_numeric_equal(pj["category_average"],qj["category_average"])
print("PASS: loaded numeric data, filter row IDs, new column, group sums/means, stable top 10, pipeline totals, and both joins agree.")


PASS: loaded numeric data, filter row IDs, new column, group sums/means, stable top 10, pipeline totals, and both joins agree.


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

In [17]:
import json, sys, datetime
cpu_model = platform.processor() or "not reported"
if Path("/proc/cpuinfo").exists():
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.startswith("model name"):
            cpu_model = line.split(":",1)[1].strip()
            break
metadata = {
    "measured_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "environment": "Assistant execution environment; re-run on the group course environment",
    "platform": platform.platform(), "cpu": cpu_model,
    "logical_cpus_visible": os.cpu_count(), "polars_threads": pl.thread_pool_size(),
    "python": platform.python_version(), "pandas": pd.__version__,
    "polars": pl.__version__, "numpy": np.__version__,
    "rows": rows, "columns": list(df_pandas.columns),
    "dataset_url": DATASET_URL, "dataset_docs": DATASET_DOCS,
    "dataset_sha256": hashlib.sha256(Path(DATA_PATH).read_bytes()).hexdigest(),
    "lazy_without_collect_ms": lazy_no_collect,
    "lazy_with_collect_ms": lazy_collected,
    "threshold_arr_delay_minutes": float(threshold),
    "repeats": 5, "pipeline_repeats": 3, "warmup_runs": 1,
    "read_cache": "warm file cache; network download excluded",
    "dtype_note": "native inferred types; Pandas nullable numeric columns are floats, Polars uses nullable integers",
    "correctness_checks": "all passed",
    "execution_method": "Fresh in-process IPython session for saved run; use Jupyter Restart Kernel and Run All locally"
}
Path("results/environment.json").write_text(json.dumps(metadata,indent=2))
Path("requirements.txt").write_text(
    f"pandas=={pd.__version__}\npolars=={pl.__version__}\nnumpy=={np.__version__}\njupyterlab>=4,<5\nipykernel>=6,<8\n")
print(json.dumps(metadata,indent=2))


{
  "measured_utc": "2026-10-09T20:58:28.628432+00:00",
  "environment": "Assistant execution environment; re-run on the group course environment",
  "platform": "Linux-6.18.44-x86_64-with-glibc2.39",
  "cpu": "AMD EPYC 9V74 80-Core Processor",
  "logical_cpus_visible": 9,
  "polars_threads": 4,
  "python": "3.12.14",
  "pandas": "2.2.3",
  "polars": "2.0.0",
  "numpy": "2.3.5",
  "rows": 336776,
  "columns": [
    "rownames",
    "year",
    "month",
    "day",
    "dep_time",
    "sched_dep_time",
    "dep_delay",
    "arr_time",
    "sched_arr_time",
    "arr_delay",
    "carrier",
    "flight",
    "tailnum",
    "origin",
    "dest",
    "air_time",
    "distance",
    "hour",
    "minute",
    "time_hour"
  ],
  "dataset_url": "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/nycflights13/flights.csv",
  "dataset_docs": "https://nycflights13.tidyverse.org/reference/flights.html",
  "dataset_sha256": "b791a2590c9167aa0bd3cb45af3f64101180878cf088c183eae4011

---
# Part 6 · Analysis draft based on the saved run

Review this draft and adapt the personal reflection to your group's experience. Timings below are from the recorded assistant environment and should be updated if you re-run on your own machine.

### Q1 — Where was the difference largest, and where was it smallest?

By the Pandas/Polars ratio, the largest difference was **new column**: Pandas took 8.262 ms and Polars took 0.763 ms (10.82x Pandas/Polars). The smallest ratio was **filter**: Pandas took 11.268 ms and Polars took 4.759 ms (2.37x Pandas/Polars). In absolute milliseconds, the largest gap was **chained pipeline** (248.260 ms) and the smallest was **filter** (6.509 ms). Creating a column can benefit from Polars reusing unchanged columns while Pandas returns another DataFrame. The chained pipeline also allows Polars to read fewer columns and apply a filter at the scan. These are explanations suggested by the code and plan, not separately measured causal effects.

### Q2 — Did anything surprise you?

The notable contrast is that creating a column showed a 10.82x ratio, while filtering showed only 2.37x. A single claim about the speed of a library would hide this variation between operations. Another important result was the lazy demonstration: plan construction took 0.022 ms, but execution took 51.917 ms. No numerical disagreement was found after aligning results; the apparent mismatch in the displayed group rows came from different group order, and stable sorting plus nulls-last made the top-ten comparison meaningful.

### Q3 — What was harder to write?

In this implementation, Pandas Boolean indexing and `assign` express the simple filter and arithmetic directly. Polars requires expression syntax such as `pl.col(...)`, but that same style carries into the lazy pipeline. The extra details needing attention were `.collect()` for execution and `nulls_last=True` with `maintain_order=True` for matching sort semantics. The relevant references are [Polars expressions](https://docs.pola.rs/user-guide/expressions/), [lazy execution](https://docs.pola.rs/user-guide/lazy/execution/) and [DataFrame.sort](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html). This is a reflection on the implementation; each group member should state which syntax they personally found clearer after trying it.

### Q4 — The marketing claim

This run reached at least 5x for **new column (10.82x), chained pipeline (10.16x), and join (6.29x)**. The other four operations did not reach 5x: reading was 4.84x, filtering 2.37x, grouping 3.46x, and sorting 3.19x. Therefore, the measurements support 5x for selected operations on this CSV, not a universal 5x claim. The result depends on row count, column types, missing values, library versions, CPU, available threads, cache state and whether end-to-end work is included. Here Polars had four worker threads, native inferred types were retained, reads used a warm cache, and each result is the median of five runs (three for the pipeline), after one warm-up. A published comparison should disclose those conditions and actually execute lazy queries. No specific published benchmark was supplied, so its exact method cannot be checked from this assignment.

### Q5 — Your recommendation

For a flight-delay reporting job that repeatedly reads this CSV, filters flights, calculates the change in delay and aggregates by airline, Polars lazy queries would be a reasonable first choice: the measured end-to-end pipeline took 27.116 ms versus 275.376 ms for the Pandas version. For a small exploratory notebook in an existing Pandas-based codebase, Pandas may be the simpler choice because the team can reuse familiar indexing and existing analysis code. The decision should also consider interoperability, memory use, maintainability and representative benchmarks; this assignment measured time, not peak memory.

### Q6 — What else is out there?

**DuckDB** — [official Python documentation](https://duckdb.org/docs/stable/clients/python/overview).

DuckDB is an embedded analytical database with a Python client that lets a program use SQL for filtering, joins and aggregations without running a separate database server. It can query CSV or Parquet files and existing Pandas DataFrames directly, which is useful for SQL-based analysis that would otherwise require importing data into another system.

### Measurement limits

One dataset and one environment are not enough to rank libraries for all projects. The measurements do not enforce identical internal dtypes, measure cold-cache I/O or peak memory, or test statistical significance; modest timing differences should not be treated as universal conclusions. Both libraries use their normal APIs, with semantic differences addressed where they affect output correctness.


---
# Submission checklist

- [x] Part 1: Polars installed and recorded in requirements.txt
- [x] Part 2: original public CSV loaded, 336,776 rows
- [x] Part 3: lazy comparison run and question answered
- [x] Part 4: all seven operations run, with observations
- [x] Part 5: table produced and results/benchmark.csv saved
- [x] Part 6: all six draft answers included; review personal wording
- [x] Every code cell executed top to bottom in a fresh IPython process, with saved outputs and passing numerical checks
- [ ] On the group computer: Restart Kernel and Run All in the course .venv; refresh written observations if timings change
- [ ] Fill all group names, student IDs, section and actual submission date
- [ ] Notebook pushed to the group's public GitHub repository
- [ ] Official Word template completed with genuine full-screen pull/push screenshots and submitted
